# Самостоятельное задание №3: Анализ пользовательских сессий (AVRO)

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
# Для чтения AVRO нужен модуль spark-avro (org.apache.spark:spark-avro_2.13:<версия Spark>).
# Здесь jar-файл spark-avro_2.13-4.3.0.jar положен в $SPARK_HOME/jars; альтернатива —
# .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.3.0")
spark = SparkSession.builder.master("local[*]").appName("Task3_Sessions").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

## Загрузка данных

In [2]:
df_raw = spark.read.format("avro").load("data/activity_log.avro")
df_raw.printSchema()
df_raw.show(truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- amount: double (nullable = true)



+--------+-------+----------+------------+-----------------+--------------------+----------+------+
|event_id|user_id|session_id|event_type  |timestamp        |page_url            |product_id|amount|
+--------+-------+----------+------------+-----------------+--------------------+----------+------+
|E001    |101    |S101-A    |login       |01-Jan-2025 10:00|/login              |NULL      |NULL  |
|E002    |101    |S101-A    |view_page   |01-Jan-2025 10:05|/products/category_A|NULL      |NULL  |
|E003    |101    |S101-A    |view_product|01-Jan-2025 10:08|/product/P005       |P005      |NULL  |
|E004    |101    |S101-A    |add_to_cart |01-Jan-2025 10:10|/cart/add           |P005      |NULL  |
|E005    |101    |S101-A    |purchase    |01-Jan-2025 10:15|/checkout           |P005      |120.5 |
|E006    |101    |S101-A    |logout      |01-Jan-2025 10:20|/logout             |NULL      |NULL  |
|E007    |102    |S102-B    |login       |02-Jan-2025 11:30|/login              |NULL      |NULL  |


## Подготовка: `timestamp` (строка) → `event_time` (TimestampType)

In [3]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
df = (df_raw
      .withColumn("event_time", F.to_timestamp("timestamp", "dd-MMM-yyyy HH:mm"))
      .drop("timestamp"))
df.printSchema()
df.show(truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



+--------+-------+----------+------------+--------------------+----------+------+-------------------+
|event_id|user_id|session_id|event_type  |page_url            |product_id|amount|event_time         |
+--------+-------+----------+------------+--------------------+----------+------+-------------------+
|E001    |101    |S101-A    |login       |/login              |NULL      |NULL  |2025-01-01 10:00:00|
|E002    |101    |S101-A    |view_page   |/products/category_A|NULL      |NULL  |2025-01-01 10:05:00|
|E003    |101    |S101-A    |view_product|/product/P005       |P005      |NULL  |2025-01-01 10:08:00|
|E004    |101    |S101-A    |add_to_cart |/cart/add           |P005      |NULL  |2025-01-01 10:10:00|
|E005    |101    |S101-A    |purchase    |/checkout           |P005      |120.5 |2025-01-01 10:15:00|
|E006    |101    |S101-A    |logout      |/logout             |NULL      |NULL  |2025-01-01 10:20:00|
|E007    |102    |S102-B    |login       |/login              |NULL      |NULL  |2

## 1. Ежедневная активность

In [4]:
df_daily = (df.groupBy(F.to_date("event_time").alias("date"))
              .agg(F.count("event_id").alias("total_events"))
              .orderBy("date"))
df_daily.show()

+----------+------------+
|      date|total_events|
+----------+------------+
|2025-01-01|           6|
|2025-01-02|           3|
|2025-01-03|           3|
|2025-01-04|           2|
|2025-01-05|           6|
|2025-01-06|           5|
|2025-01-07|           5|
|2025-01-08|           4|
|2025-01-09|           6|
+----------+------------+



## 2. Анализ пользователей и сессий

In [5]:
print("Количество уникальных пользователей")
df.select("user_id").distinct().orderBy("user_id").show()

print("Количество уникальных сессий")
print(df.select("user_id", "session_id").distinct().count())

df.groupBy("user_id") \
  .agg(F.countDistinct("session_id").alias("unique_sessions_count")) \
  .orderBy("user_id").show()

Количество уникальных пользователей


+-------+
|user_id|
+-------+
|    101|
|    102|
|    103|
|    104|
|    105|
|    106|
|    107|
+-------+

Количество уникальных сессий


9


+-------+---------------------+
|user_id|unique_sessions_count|
+-------+---------------------+
|    101|                    2|
|    102|                    2|
|    103|                    1|
|    104|                    1|
|    105|                    1|
|    106|                    1|
|    107|                    1|
+-------+---------------------+



## 3. Детализация сессий

In [6]:
(df.groupBy("user_id", "session_id")
   .agg(F.count("event_id").alias("count_events_in_session"))
   .orderBy("count_events_in_session", "user_id", "session_id")
   .show())

+-------+----------+-----------------------+
|user_id|session_id|count_events_in_session|
+-------+----------+-----------------------+
|    104|    S104-D|                      2|
|    102|    S102-B|                      3|
|    103|    S103-C|                      3|
|    106|    S106-F|                      4|
|    101|    S101-B|                      5|
|    102|    S102-C|                      5|
|    101|    S101-A|                      6|
|    105|    S105-E|                      6|
|    107|    S107-G|                      6|
+-------+----------+-----------------------+



## 4. Анализ покупок

In [7]:
purchases = df.filter(F.col("event_type") == "purchase")
purchases.agg(F.count("*").alias("count_purchase_amount")).show()
purchases.agg(F.sum("amount").alias("sum_purchase_amount")).show()
purchases.agg(F.round(F.avg("amount"), 2).alias("average_purchase_amount")).show()

+---------------------+
|count_purchase_amount|
+---------------------+
|                    4|
+---------------------+

+-------------------+
|sum_purchase_amount|
+-------------------+
|              905.5|
+-------------------+



+-----------------------+
|average_purchase_amount|
+-----------------------+
|                 226.38|
+-----------------------+



## 5. Анализ продолжительности сессий

In [8]:
df_sessions = (df.groupBy("user_id", "session_id")
    .agg(F.min("event_time").alias("min_time"), F.max("event_time").alias("max_time"))
    .withColumn("session_duration_seconds",
                F.unix_timestamp("max_time") - F.unix_timestamp("min_time"))
    .orderBy("user_id", "session_id"))
df_sessions.show(truncate=False)

df_sessions.agg(F.round(F.avg("session_duration_seconds"), 2)
                .alias("average_session_duration_seconds")).show()

+-------+----------+-------------------+-------------------+------------------------+
|user_id|session_id|min_time           |max_time           |session_duration_seconds|
+-------+----------+-------------------+-------------------+------------------------+
|101    |S101-A    |2025-01-01 10:00:00|2025-01-01 10:20:00|1200                    |
|101    |S101-B    |2025-01-06 09:30:00|2025-01-06 09:40:00|600                     |
|102    |S102-B    |2025-01-02 11:30:00|2025-01-02 11:38:00|480                     |
|102    |S102-C    |2025-01-07 15:00:00|2025-01-07 15:18:00|1080                    |
|103    |S103-C    |2025-01-03 09:00:00|2025-01-03 09:10:00|600                     |
|104    |S104-D    |2025-01-04 14:00:00|2025-01-04 14:03:00|180                     |
|105    |S105-E    |2025-01-05 16:45:00|2025-01-05 17:02:00|1020                    |
|106    |S106-F    |2025-01-08 10:00:00|2025-01-08 10:10:00|600                     |
|107    |S107-G    |2025-01-09 11:00:00|2025-01-09 11:

+--------------------------------+
|average_session_duration_seconds|
+--------------------------------+
|                           740.0|
+--------------------------------+



In [9]:
spark.stop()